# 02 · MIDI 카피 검사
원본: check/_Does_it_good.ipynb

팀원들이 직접 만든 단일 트랙 MIDI를 피아노롤로 확인하는 도구입니다. 세로축은 음높이, 가로축 한 칸은 16분음표입니다. 빈 파일이나 여러 음악 트랙은 지원하지 않습니다. 변환 함수는 당시 구현을 보존합니다.

In [ ]:
!pip install mido
from mido import MidiFile
import numpy as np
import glob
import random
import matplotlib.pyplot as plt
from IPython import display

In [ ]:
from google.colab import files
uploaded = files.upload()
path = f'/content/{list(uploaded.keys())[0]}'

In [ ]:
#@title

def read_midi_file(midi_file_path, ticks_per_beat=480):
    # print('midi_file_path', midi_file_path)
    mid = MidiFile(midi_file_path, ticks_per_beat=ticks_per_beat)
    note_temp = []
    note_import = []
    tick = 0
    for i, track in enumerate(mid.tracks):
        for msg in track:
            tick += msg.time
            if msg.type == 'note_on' or msg.type == 'note_off':
                if msg.type == 'note_on' and msg.velocity !=0:
                    pitch = msg.note
                    start = tick*4 / float(mid.ticks_per_beat)
                    start = float("{:.4f}".format(start))  # 시간 퀀타이즈
                    velocity = msg.velocity
                    note_info = {
                        'pitch': pitch,
                        'start_time': start,
                        'length': None,
                        'velocity': velocity,
#                         'function': None,
                    }
                    note_temp.append(note_info)
                elif msg.type == 'note_off' or msg.velocity == 0:
                    for j in range(len(note_temp)):
                        if note_temp[j]['pitch'] == msg.note:
                            end = tick*4 / float(mid.ticks_per_beat)
                            note_temp[j]['length'] = end - note_temp[j]['start_time']
                            note_import.append(note_temp[j])
                            del note_temp[j]
                            break
            else:
                pass
                # print(msg)

    note_import = sorted(note_import, key=lambda k: k['start_time'])
    return note_import

#제거할 피치 리스트
remove_these_rows = []
for i in range(36):
  remove_these_rows.append(i)
for i in range(60,72):
  remove_these_rows.append(i)
for i in range(112, 128):
  remove_these_rows.append(i)

def ploting():
  song = read_midi_file(path)
  temp_data = np.zeros((128,128))
  if song[-1]['start_time'] >= 128: #9마디 짜리이면
      for note in song:
        if not note['start_time'] < 16.0: #못 갖춘 마디 버림
          x = int(note['pitch'])
          y = int(round(note['start_time']))
          length = int(round(note['length']))
          for i in range(length):
            if y+i-16 > 127:
                False
            else:
              temp_data[x][y+i-16] = 1
  else: #8마디 짜리이면
      for note in song:
        x = int(note['pitch'])
        y = int(round(note['start_time']))
        length = int(round(note['length']))
        for i in range(length):
          if y+i > 127:
              pass
          else:
            temp_data[x][y+i] = 1


  # temp_data = np.delete(temp_data, remove_these_rows, axis=0)
  # temp_data1 = np.delete(temp_data, np.s_[64::1], axis=1)
  # temp_data2 = np.delete(temp_data, np.s_[:64:1], axis=1)
  # print(temp_data1.shape, temp_data2.shape)

  display.clear_output(wait=True)

  fig = plt.figure(figsize=(10,10))
  plt.xlim([0, 127])      # X축의 범위: [xmin, xmax]
  plt.ylim([0, 127])     # Y축의 범위: [ymin, ymax]
  plt.xticks([0, 63, 127])
  plt.yticks([0, 35, 60, 71, 112, 127])
  plt.grid(True)
  display.clear_output(wait=True)
      
  plt.imshow(temp_data[:, :] * 127.5 + 127.5, cmap='gray', origin='lower')

In [ ]:
ploting()